### CSV and EXCEL Files - Structured Data

In [6]:
import pandas

### CSV processing ###

In [7]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader

C:\Users\klinton\AppData\Local\Temp\ipykernel_36892\444435838.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader
d:\RAG\RAG-Learning\Repo-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# mEthod 1: Using CSVLoader
print("Method 1: Using CSVLoader")
# Load the CSV file using CSVLoader
csv_loader = CSVLoader(file_path='data/csv/annual.csv', encoding='utf-8',csv_args={'delimiter': ',', 'quotechar': '"'})

csv_documents = csv_loader.load()
# Print the loaded documents
print(f"Number of documents loaded: {len(csv_documents)}")
print(f"First document content:\n{csv_documents[0].page_content}\n")
print(f"First document metadata:\n{csv_documents[0].metadata}\n")

Method 1: Using CSVLoader
Number of documents loaded: 23220
First document content:
﻿year: 2011
industry_code_ANZSIC: A
industry_name_ANZSIC: Agriculture, Forestry and Fishing
rme_size_grp: a_0
variable: Activity unit
value: 46134
unit: COUNT
: 

First document metadata:
{'source': 'data/csv/annual.csv', 'row': 0}



In [9]:
from typing import List
from langchain_core.documents import Document

#method 2: Custom CSV Processing
print("Method 2: Custom CSV Processing")
def process_csv_intelligently(file_path: str) -> List[Document]:
    # Read the CSV file into a DataFrame
    df = pandas.read_csv(file_path)
    
    documents = []
    for index, row in df.iterrows():
        # Create a document for each row
        content = f"""Annual income statement for {row['industry_name_ANZSIC']} ({row['year']}):
                    Revenue: {row['rme_size_grp']}"""

        document = Document(
            page_content=content,
            metadata={"row_index": index, "company": row['industry_name_ANZSIC'], "year": row['year'], "revenue": row['rme_size_grp'], "data_type": "csv"}
        )
        documents.append(document)
    return documents


Method 2: Custom CSV Processing


In [10]:
processed_documents = process_csv_intelligently('data/csv/annual.csv')
# Print the processed documents
for doc in processed_documents:
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

Content: Annual income statement for Agriculture, Forestry and Fishing (2011):
                    Revenue: a_0
Metadata: {'row_index': 0, 'company': 'Agriculture, Forestry and Fishing', 'year': 2011, 'revenue': 'a_0', 'data_type': 'csv'}
Content: Annual income statement for Agriculture, Forestry and Fishing (2011):
                    Revenue: a_0
Metadata: {'row_index': 1, 'company': 'Agriculture, Forestry and Fishing', 'year': 2011, 'revenue': 'a_0', 'data_type': 'csv'}
Content: Annual income statement for Agriculture, Forestry and Fishing (2011):
                    Revenue: a_0
Metadata: {'row_index': 2, 'company': 'Agriculture, Forestry and Fishing', 'year': 2011, 'revenue': 'a_0', 'data_type': 'csv'}
Content: Annual income statement for Agriculture, Forestry and Fishing (2011):
                    Revenue: a_0
Metadata: {'row_index': 3, 'company': 'Agriculture, Forestry and Fishing', 'year': 2011, 'revenue': 'a_0', 'data_type': 'csv'}
Content: Annual income statement for Agricul

### Excel Processing ###

In [22]:
print("Pandas Based Excel processing")

def process_excel_with_pandas(file_path: str) -> List[Document]:
    documents = []

    # Read the Excel file
    df = pandas.ExcelFile(file_path)

    # Process each sheet
    for sheet_name in df.sheet_names:

        # Read the current sheet into a DataFrame
        sheet_df = pandas.read_excel(
            file_path,
            sheet_name=sheet_name
        )

        # Create document content for the sheet
        sheet_content = f"Sheet: {sheet_name}\n"
        sheet_content += f"Columns: {', '.join(sheet_df.columns.astype(str))}\n"
        sheet_content += f"Number of rows: {len(sheet_df)}\n\n"

        # Add actual Excel data
        sheet_content += sheet_df.to_string(index=False)

        # Create LangChain Document
        doc = Document(
            page_content=sheet_content,
            metadata={
                "source": file_path,
                "sheet_name": sheet_name,
                "num_rows": len(sheet_df),
                "num_columns": len(sheet_df.columns),
                "data_type": "excel"
            }
        )

        documents.append(doc)

    return documents

Pandas Based Excel processing


In [23]:
excel_documents = process_excel_with_pandas('data/excel/people.xls')

print(f"Number of documents loaded from Excel: {len(excel_documents)}")
print(excel_documents)

Number of documents loaded from Excel: 1
[Document(metadata={'source': 'data/excel/people.xls', 'sheet_name': 'Sheet1', 'num_rows': 100, 'num_columns': 8, 'data_type': 'excel'}, page_content='Sheet: Sheet1\nColumns: 0, First Name, Last Name, Gender, Country, Age, Date, Id\nNumber of rows: 100\n\n  0 First Name  Last Name Gender       Country  Age       Date   Id\n  1      Dulce      Abril Female United States   32 15/10/2017 1562\n  2       Mara  Hashimoto Female Great Britain   25 16/08/2016 1582\n  3     Philip       Gent   Male        France   36 21/05/2015 2587\n  4   Kathleen     Hanner Female United States   25 15/10/2017 3549\n  5    Nereida    Magwood Female United States   58 16/08/2016 2468\n  6     Gaston      Brumm   Male United States   24 21/05/2015 2554\n  7       Etta       Hurn Female Great Britain   56 15/10/2017 3598\n  8    Earlean     Melgar Female United States   27 16/08/2016 2456\n  9   Vincenza    Weiland Female United States   40 21/05/2015 6548\n 10     Fallo

In [ ]:
print(f"Unstructured Excel processing")

try:
    from langchain_community.document_loaders import UnstructuredExcelLoader

    # Load the Excel file using UnstructuredExcelLoader
    unstructured_excel_loader = UnstructuredExcelLoader(file_path='data/excel/people.xls',mode="elements")
    unstructured_excel_documents = unstructured_excel_loader.load()

    # Print the loaded documents
    print(f"Number of documents loaded from UnstructuredExcelLoader: {len(unstructured_excel_documents)}")
    print(f"First document content:\n{unstructured_excel_documents[0].page_content}\n")
    print(f"First document metadata:\n{unstructured_excel_documents[0].metadata}\n")
except ImportError as e:
    print("UnstructuredExcelLoader is not available. Please ensure you have the required dependencies installed.")
    print(f"Error: {e}")